In [ ]:
# Install qcirclab from the repository.
# In a local environment you may prefer: pip install -e /path/to/qcirclab_repo
!pip install -q git+https://github.com/2forts/qcirclab_repo.git

In [ ]:
import time
import math
import random
from collections import deque, Counter

import numpy as np

from qcirclab import (
    Circuit,
    circuit_metrics,
    print_metrics,
    append_operation,
    circuit_without_measurements,
    circuit_unitary,
    equal_up_to_global_phase,
)

import qcirclab.gates as qg

from qcirclab.passes import (
    AnalysisPass,
    TransformationPass,
    PassPipeline,
)

# Subsection **8.2.4 Implementing global passes with qcirclab**

In [ ]:
class GlobalCostAnalysis(AnalysisPass):
    def __init__(self, alpha=1.0, beta=1.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta

    def run(self, qc: Circuit) -> Circuit:
        metrics = circuit_metrics(qc)
        depth = metrics["depth"]
        twoq_gates = metrics["two_qubit_gates"]
        cost = self.alpha * depth + self.beta * twoq_gates
        self.property_set["global_cost"] = cost
        self.property_set["global_metrics"] = metrics
        return qc

In [ ]:
SELF_INVERSE = {"h", "x", "y", "z", "cx", "cz", "swap"}


def same_location(op1, op2):
    return (
        op1.name == op2.name
        and tuple(op1.targets) == tuple(op2.targets)
        and tuple(op1.controls) == tuple(op2.controls)
        and op1.condition == op2.condition
    )


def cancel_adjacent_gates(qc: Circuit) -> Circuit:
    stack = []

    for op in qc.operations:
        if op.name == "barrier":
            continue

        if stack:
            prev = stack[-1]

            if (
                op.name in SELF_INVERSE
                and prev.name in SELF_INVERSE
                and same_location(prev, op)
            ):
                stack.pop()
                continue

            if (
                op.name in {"rx", "ry", "rz"}
                and prev.name == op.name
                and tuple(prev.targets) == tuple(op.targets)
                and tuple(prev.controls) == tuple(op.controls)
                and prev.params
                and op.params
                and abs(prev.params[0] + op.params[0]) < 1e-12
            ):
                stack.pop()
                continue

        stack.append(op)

    out = Circuit(qc.n_qubits, qc.n_clbits, name=qc.name + "_cancelled")
    for op in stack:
        append_operation(out, op)
    return out


def propose_move(qc: Circuit) -> Circuit:
    return cancel_adjacent_gates(qc)

In [ ]:
class SimulatedAnnealingStep(TransformationPass):
    def __init__(self, alpha=1.0, beta=1.0, temperature=1.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.temperature = temperature

    def _cost(self, qc: Circuit):
        metrics = circuit_metrics(qc)
        return self.alpha * metrics["depth"] + self.beta * metrics["two_qubit_gates"]

    def run(self, qc: Circuit) -> Circuit:
        current_cost = self._cost(qc)
        candidate = propose_move(qc)
        candidate_cost = self._cost(candidate)

        if candidate_cost < current_cost:
            return candidate

        delta = candidate_cost - current_cost
        prob = math.exp(-delta / max(self.temperature, 1e-8))
        if random.random() < prob:
            return candidate
        return qc

# Shared basis-decomposition helper

In [ ]:
def decompose_to_rx_rz_cx(qc: Circuit) -> Circuit:
    out = Circuit(qc.n_qubits, qc.n_clbits, name=qc.name + "_basis")

    for op in qc.operations:
        if op.name in {"barrier", "measure", "reset"}:
            append_operation(out, op)

        elif op.name == "h":
            q = op.targets[0]
            out.rz(np.pi/2, q)
            out.rx(np.pi/2, q)
            out.rz(np.pi/2, q)

        elif op.name == "x":
            out.rx(np.pi, op.targets[0])

        elif op.name == "z":
            out.rz(np.pi, op.targets[0])

        elif op.name == "s":
            out.rz(np.pi/2, op.targets[0])

        elif op.name == "t":
            out.rz(np.pi/4, op.targets[0])

        elif op.name == "tdg":
            out.rz(-np.pi/4, op.targets[0])

        elif op.name == "ry":
            q = op.targets[0]
            theta = op.params[0]
            out.rz(np.pi/2, q)
            out.rx(theta, q)
            out.rz(-np.pi/2, q)

        elif op.name == "cz":
            c = op.controls[0]
            t = op.targets[0]
            out.rz(np.pi/2, t).rx(np.pi/2, t).rz(np.pi/2, t)
            out.cx(c, t)
            out.rz(np.pi/2, t).rx(np.pi/2, t).rz(np.pi/2, t)

        elif op.name in {"rx", "rz", "cx"}:
            append_operation(out, op)

        else:
            append_operation(out, op)

    return out

In [ ]:
def generate_candidates(qc: Circuit):
    basis = decompose_to_rx_rz_cx(qc)

    return [
        ("original", qc),
        ("cancel_adjacent", cancel_adjacent_gates(qc)),
        ("basis_rx_rz_cx", basis),
        ("basis_then_cancel", cancel_adjacent_gates(basis)),
    ]


class BestGlobalCandidate(TransformationPass):
    def __init__(self, alpha=1.0, beta=2.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta

    def _cost(self, qc: Circuit):
        metrics = circuit_metrics(qc)
        return (
            self.alpha * metrics["depth"]
            + self.beta * metrics["two_qubit_gates"]
        )

    def run(self, qc: Circuit) -> Circuit:
        candidates = generate_candidates(qc)

        scored = []
        for name, cand in candidates:
            cost = self._cost(cand)
            metrics = circuit_metrics(cand)
            scored.append((cost, name, cand, metrics))

        best_cost, best_name, best_circuit, best_metrics = min(
            scored,
            key=lambda x: x[0],
        )

        self.property_set["candidate_costs"] = [
            {
                "name": name,
                "cost": cost,
                "metrics": metrics,
            }
            for cost, name, _, metrics in scored
        ]
        self.property_set["best_candidate"] = best_name
        self.property_set["best_candidate_cost"] = best_cost

        return best_circuit

In [ ]:
def encode_circuit(qc: Circuit):
    m = circuit_metrics(qc)
    return np.array([
        m["operations"],
        m["depth"],
        m["two_qubit_gates"],
        m["multi_qubit_gates"],
    ], dtype=float)


class SimpleLinearModel:
    def __init__(self, weights=(0.2, 1.0, 1.5, 3.0)):
        self.weights = np.asarray(weights, dtype=float)

    def predict(self, features):
        return float(np.dot(self.weights, features))


class LearnedGlobalMove(TransformationPass):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def run(self, qc: Circuit) -> Circuit:
        candidates = generate_candidates(qc)

        scored = []
        for name, cand in candidates:
            features = encode_circuit(cand)
            score = self.model.predict(features)
            scored.append((score, name, cand))

        best_score, best_name, best_circuit = min(scored, key=lambda x: x[0])

        self.property_set["learned_move_score"] = best_score
        self.property_set["learned_move"] = best_name

        return best_circuit

In [ ]:
pm = PassPipeline([
    BestGlobalCandidate(alpha=1.0, beta=2.0),
    GlobalCostAnalysis(alpha=1.0, beta=2.0),
])

qc = Circuit(2)
qc.h(0)
qc.h(0)
qc.cx(0, 1)
qc.cx(0, 1)
qc.rz(0.3, 1)
qc.rz(-0.3, 1)
qc.x(0)
qc.x(0)
qc.cz(0, 1)
qc.cz(0, 1)

new_qc = pm.run(qc)

print("Original:")
print(qc.draw())
print_metrics("Original", qc)

print("After global candidate selection:")
print(new_qc.draw())
print_metrics("Selected circuit", new_qc)

print("Property set:")
print(pm.property_set)

# Subsection **8.3.5 Custom rewrite rules and equivalence checking**

In [ ]:
class SimpleEquivalenceLibrary:
    def __init__(self):
        self._rules = {}

    def add_equivalence(self, gate_name, builder):
        self._rules.setdefault(gate_name, []).append(builder)

    def get_entry(self, gate_name):
        return self._rules.get(gate_name, [])


def cz_to_hcxh(c, t):
    decomp = Circuit(2)
    decomp.h(1)
    decomp.cx(0, 1)
    decomp.h(1)
    return decomp


equiv_lib = SimpleEquivalenceLibrary()
equiv_lib.add_equivalence("cz", cz_to_hcxh)


class CZToCXRewrite(TransformationPass):
    def __init__(self, equiv_lib):
        super().__init__()
        self.equiv_lib = equiv_lib

    def run(self, qc: Circuit) -> Circuit:
        out = Circuit(qc.n_qubits, qc.n_clbits, name=qc.name + "_cz_rewritten")

        for op in qc.operations:
            if op.name == "cz":
                c = op.controls[0]
                t = op.targets[0]
                out.h(t)
                out.cx(c, t)
                out.h(t)
            else:
                append_operation(out, op)

        return out

In [ ]:
qc_cz = Circuit(2)
qc_cz.h(0)
qc_cz.cz(0, 1)
qc_cz.rz(0.2, 1)

rewrite = CZToCXRewrite(equiv_lib)
qc_rewritten = rewrite.run(qc_cz)

print("Original:")
print(qc_cz.draw())
print_metrics("Original", qc_cz)

print("Rewritten:")
print(qc_rewritten.draw())
print_metrics("Rewritten", qc_rewritten)

print(
    "Equivalent up to global phase:",
    equal_up_to_global_phase(circuit_unitary(qc_cz), circuit_unitary(qc_rewritten))
)

# Subsection **8.4.4 Timing-aware profiling with qcriclab utilities**

In [ ]:
gate_durations_ns = {
    "h": 35,
    "x": 35,
    "rx": 35,
    "ry": 35,
    "rz": 0,
    "s": 0,
    "t": 0,
    "tdg": 0,
    "cx": 300,
    "cz": 300,
    "swap": 900,
    "measure": 1000,
}


def schedule_asap(qc: Circuit, durations=gate_durations_ns):
    qtime = [0.0] * qc.n_qubits
    ctime = [0.0] * qc.n_clbits
    rows = []

    for op in qc.operations:
        if op.name == "barrier":
            m = max(qtime + ctime) if (qtime or ctime) else 0.0
            qtime = [m] * qc.n_qubits
            ctime = [m] * qc.n_clbits
            rows.append((op.name, (), (), m, m))
            continue

        used_q = set(op.targets) | set(op.controls)
        used_c = set(getattr(op, "ctargets", ()))
        if getattr(op, "condition", None) is not None:
            used_c.add(op.condition.bit)

        start = 0.0
        if used_q:
            start = max(start, max(qtime[q] for q in used_q))
        if used_c:
            start = max(start, max(ctime[c] for c in used_c))

        duration = durations.get(op.name, 50)
        finish = start + duration

        for q in used_q:
            qtime[q] = finish
        for c in used_c:
            ctime[c] = finish

        rows.append((
            op.name,
            tuple(sorted(used_q)),
            tuple(sorted(used_c)),
            start,
            finish,
        ))

    total_duration = max(qtime + ctime) if (qtime or ctime) else 0.0
    return rows, total_duration


def print_schedule(rows):
    for name, qs, cs, start, finish in rows:
        print(
            f"{name:8s} q={qs} c={cs} "
            f"start={start:7.1f} ns "
            f"finish={finish:7.1f} ns"
        )

In [ ]:
def idle_windows(rows, n_qubits, total_duration):
    active = [[] for _ in range(n_qubits)]

    for name, qs, cs, start, finish in rows:
        if finish <= start:
            continue
        for q in qs:
            active[q].append((start, finish, name))

    windows = {q: [] for q in range(n_qubits)}

    for q, intervals in enumerate(active):
        intervals.sort()
        current = 0.0

        for start, finish, _ in intervals:
            if start > current:
                windows[q].append((current, start))
            current = max(current, finish)

        if total_duration > current:
            windows[q].append((current, total_duration))

    return windows


def print_idle_windows(windows):
    for q in sorted(windows):
        pieces = [
            f"[{start:.1f}, {finish:.1f})"
            for start, finish in windows[q]
        ]
        text = ", ".join(pieces) if pieces else "no idle windows"
        print(f"q{q}: {text}")

In [ ]:
qc_free = Circuit(3)
qc_free.h(0)
qc_free.cx(0, 1)
qc_free.x(2)
qc_free.cx(1, 2)

qc_barrier = Circuit(3)
qc_barrier.h(0)
qc_barrier.cx(0, 1)
qc_barrier.barrier()
qc_barrier.x(2)
qc_barrier.cx(1, 2)

for label, qc in [
    ("Without barrier", qc_free),
    ("With barrier", qc_barrier),
]:
    rows, duration = schedule_asap(qc)
    windows = idle_windows(rows, qc.n_qubits, duration)

    print("\n===", label, "===")
    print(qc.draw())
    print_schedule(rows)
    print("ASAP duration (ns):", duration)
    print("Idle windows:")
    print_idle_windows(windows)

# Subsection **8.5.4 Logical resource estimation with circuit utilities**

In [ ]:
import numpy as np

T_MATRIX = np.array([
    [1.0, 0.0],
    [0.0, np.exp(1j * np.pi / 4)],
], dtype=complex)

TDG_MATRIX = T_MATRIX.conj().T


def add_t(qc: Circuit, q: int):
    if hasattr(qc, "t"):
        return qc.t(q)
    return qc.unitary(T_MATRIX, [q], name="t")


def add_tdg(qc: Circuit, q: int):
    if hasattr(qc, "tdg"):
        return qc.tdg(q)
    return qc.unitary(TDG_MATRIX, [q], name="tdg")

In [ ]:
T_GATES = {"t", "tdg"}


def op_qubits(op, n_qubits):
    if op.name == "barrier":
        return set(range(n_qubits))

    return set(op.targets) | set(op.controls)


def t_demand_profile(qc: Circuit):
    # layer[q] is the earliest T-layer index available on qubit q.
    layer = [0] * qc.n_qubits

    profile = {}
    t_events = []

    for index, op in enumerate(qc.operations):
        qs = op_qubits(op, qc.n_qubits)

        if not qs:
            continue

        current = max(layer[q] for q in qs)

        if op.name in T_GATES:
            profile[current] = profile.get(current, 0) + 1
            t_events.append((
                index,
                op.name,
                tuple(sorted(qs)),
                current,
            ))

            for q in qs:
                layer[q] = current + 1
        else:
            # Clifford or synchronization operation:
            # it does not consume a T layer, but it propagates dependencies.
            for q in qs:
                layer[q] = current

    if not profile:
        return [], t_events

    max_layer = max(profile)
    demand = [profile.get(k, 0) for k in range(max_layer + 1)]
    return demand, t_events

In [ ]:
def logical_t_summary(qc: Circuit):
    demand, t_events = t_demand_profile(qc)

    t_count = sum(demand)
    t_depth = len(demand)
    peak_demand = max(demand, default=0)

    return {
        "t_count": t_count,
        "t_depth": t_depth,
        "t_demand_profile": demand,
        "peak_t_demand": peak_demand,
        "t_events": t_events,
    }


def print_logical_t_summary(label, qc):
    summary = logical_t_summary(qc)

    print(f"=== {label} ===")
    print("T-count:", summary["t_count"])
    print("T-depth:", summary["t_depth"])
    print("T-demand profile:", summary["t_demand_profile"])
    print("Peak T demand:", summary["peak_t_demand"])

    print("T events:")
    for index, name, qs, layer in summary["t_events"]:
        print(
            f"  op {index:2d}: {name:3s} "
            f"q={qs} layer={layer}"
        )

    return summary

In [ ]:
def factory_pressure(summary, factories=1, rate_per_factory=1):
    demand = summary["t_demand_profile"]
    capacity = factories * rate_per_factory

    t_count = summary["t_count"]
    peak = summary["peak_t_demand"]

    lower_bound_supply_steps = (
        int(np.ceil(t_count / capacity)) if capacity > 0 else np.inf
    )

    factories_for_peak = (
        int(np.ceil(peak / rate_per_factory))
        if rate_per_factory > 0
        else np.inf
    )

    layer_shortages = []
    for layer, requested in enumerate(demand):
        shortage = max(0, requested - capacity)
        if shortage:
            layer_shortages.append((layer, requested, shortage))

    max_buffer_needed = max(
        (shortage for _, _, shortage in layer_shortages),
        default=0,
    )

    return {
        "factories": factories,
        "rate_per_factory": rate_per_factory,
        "capacity_per_layer": capacity,
        "factory_steps_lower_bound": lower_bound_supply_steps,
        "factories_needed_for_peak": factories_for_peak,
        "layer_shortages": layer_shortages,
        "max_buffer_needed": max_buffer_needed,
    }


def print_factory_pressure(summary, factories=1, rate_per_factory=1):
    pressure = factory_pressure(
        summary,
        factories=factories,
        rate_per_factory=rate_per_factory,
    )

    print("Factory model:")
    print("  factories:", pressure["factories"])
    print("  rate per factory:", pressure["rate_per_factory"])
    print("  capacity per T-layer:", pressure["capacity_per_layer"])
    print(
        "  lower bound on factory production steps:",
        pressure["factory_steps_lower_bound"],
    )
    print(
        "  factories needed to meet peak demand:",
        pressure["factories_needed_for_peak"],
    )
    print(
        "  buffered states needed with this factory count:",
        pressure["max_buffer_needed"],
    )

    if pressure["layer_shortages"]:
        print("  shortage layers:")
        for layer, requested, shortage in pressure["layer_shortages"]:
            print(
                f"    layer {layer}: "
                f"requested={requested}, shortage={shortage}"
            )
    else:
        print("  no instantaneous shortages")

In [ ]:
def burst_t_block(n=6):
    qc = Circuit(n, name="burst_T_block")
    for q in range(n):
        add_t(qc, q)
    return qc


def serialized_t_block(n=6):
    qc = Circuit(n, name="serialized_T_block")
    for q in range(n):
        add_t(qc, q)
        if q != n - 1:
            qc.barrier()
    return qc


qc_burst = burst_t_block(6)
qc_serial = serialized_t_block(6)

for label, qc in [
    ("Burst T block", qc_burst),
    ("Serialized T block", qc_serial),
]:
    print()
    print(qc.draw())

    summary = print_logical_t_summary(label, qc)
    print_factory_pressure(
        summary,
        factories=1,
        rate_per_factory=2,
    )

# Subsection **8.6.2 End-to-end example and diagnostics**

In [ ]:
qc_input = Circuit(3, name="workflow_input")

qc_input.h(0)
add_t(qc_input, 0)
qc_input.cx(0, 1)
qc_input.rz(0.3, 1)
qc_input.cz(1, 2)
qc_input.x(2)
qc_input.x(2)

print("Input circuit:")
print(qc_input.draw())
print_metrics("Input", qc_input)

In [ ]:
def candidate_original(qc):
    return qc


def candidate_cleanup(qc):
    return cancel_adjacent_gates(qc)


def candidate_cx_basis(qc):
    rewritten = CZToCXRewrite(equiv_lib).run(qc)
    lowered = decompose_to_rx_rz_cx(rewritten)
    cleaned = cancel_adjacent_gates(lowered)
    return cleaned


candidate_builders = {
    "original": candidate_original,
    "cleanup": candidate_cleanup,
    "cx_basis": candidate_cx_basis,
}

In [ ]:
def evaluate_workflow_candidate(name, qc_original, qc_candidate):
    equivalent = equal_up_to_global_phase(
        circuit_unitary(qc_original),
        circuit_unitary(qc_candidate),
    )

    metrics = circuit_metrics(qc_candidate)

    schedule, duration_ns = schedule_asap(qc_candidate)

    logical = logical_t_summary(qc_candidate)

    return {
        "name": name,
        "circuit": qc_candidate,
        "equivalent": equivalent,
        "metrics": metrics,
        "duration_ns": duration_ns,
        "logical": logical,
    }


candidate_reports = {}

for name, builder in candidate_builders.items():
    qc_candidate = builder(qc_input)
    candidate_reports[name] = evaluate_workflow_candidate(
        name,
        qc_input,
        qc_candidate,
    )

In [ ]:
def print_workflow_report(report):
    metrics = report["metrics"]
    logical = report["logical"]

    print(f"=== {report['name']} ===")
    print("Equivalent:", report["equivalent"])
    print("operations:", metrics["operations"])
    print("depth:", metrics["depth"])
    print("two-qubit gates:", metrics["two_qubit_gates"])
    print("scheduled duration (ns):", report["duration_ns"])
    print("T-count:", logical["t_count"])
    print("T-depth:", logical["t_depth"])
    print("T-demand profile:", logical["t_demand_profile"])
    print()


for report in candidate_reports.values():
    print_workflow_report(report)

In [ ]:
def nisq_score(report):
    m = report["metrics"]
    return (
        m["depth"]
        + 2.0 * m["two_qubit_gates"]
        + 0.01 * report["duration_ns"]
    )


def logical_score(report):
    m = report["metrics"]
    logical = report["logical"]
    return (
        10.0 * logical["t_count"]
        + 5.0 * logical["t_depth"]
        + 0.5 * m["two_qubit_gates"]
    )


def select_report(reports, score_fn):
    scored = [
        (score_fn(report), name)
        for name, report in reports.items()
        if report["equivalent"]
    ]
    return sorted(scored)


print("NISQ-oriented ranking:")
for score, name in select_report(candidate_reports, nisq_score):
    print(f"  {name:10s} score={score}")

print("\nFault-tolerant-oriented ranking:")
for score, name in select_report(candidate_reports, logical_score):
    print(f"  {name:10s} score={score}")